# Monitoring and Alerting: Logistics SLA

This notebook continuously tracks fulfillment performance and delay rates over time.

### Alerting Strategy:
1. **TPC-H Baseline Context:** The synthetic TPC-H generator naturally produces delivery dates resulting in a baseline delay rate around ~63%. A naive static threshold (e.g., 15%) causes persistent false positives.
2. **Absolute Threshold (> 65.0%):** Flags periods where the delay rate significantly exceeds the historical normal range.
3. **MoM Spike Detection (> 5.0 pp):** Alerts on sudden month-over-month increases (>5 percentage points) in the delay rate, highlighting unexpected operational degradation.

When integrated into Databricks Jobs, an unhandled exception generated here automatically sends an alert notification (Email/Slack) to logistics stakeholders.

In [0]:
import os
import sys
from pathlib import Path

os.environ["DB_CATALOG"] = "workspace"
os.environ["SCHEMA_PREFIX"] = "tpch"

repo_root = Path.cwd().parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from config import env
from src.monitoring.delay_alerts import run_delay_monitoring

print(f"Running delay monitoring pipeline on {env.CATALOG}.{env.SILVER_SCHEMA}...")

# Thresholds: >65% absolute delay rate OR >5 percentage points MoM spike
monitoring_results = run_delay_monitoring(
    spark,
    absolute_threshold_pct=65.0,
    mom_spike_threshold=5.0
)

metrics_df = monitoring_results["metrics"]
alerts_df = monitoring_results["alerts"]
print("Monthly metrics and alert conditions calculated successfully.")

## 1. Monthly Delay Rate Dynamics
The table below displays monthly line item counts, delay percentages, and month-over-month (MoM) rate changes.

In [0]:
display(metrics_df)

Databricks visualization. Run in Databricks to view.

## 2. Trigger Evaluation (Breached Periods)
Filters for months that breached either the 65.0% absolute boundary or the 5.0 pp MoM spike threshold. 

The cell below validates the alert count and raises an `AssertionError` if critical breaches are detected, failing the Databricks Job as intended for alerting.

In [0]:
display(alerts_df)

alert_count = alerts_df.count()

if alert_count > 0:
    breached_periods = [
        f"{row['receipt_month'].strftime('%Y-%m')} (Rate: {row['delay_rate_pct']}%, ΔMoM: {row['rate_change_pct_points']} pp)"
        for row in alerts_df.collect()
    ]
    
    raise AssertionError(
        f"🚨 ALERT: Detected {alert_count} period(s) breaching logistics SLA thresholds "
        f"(Rate > 65.0% or MoM Spike > 5.0 pp)!\n"
        f"Breached periods:\n - " + "\n - ".join(breached_periods) + "\n"
        f"Investigation required by the Logistics Operations team."
    )
else:
    print("✅ All periods within SLA tolerances. No unexpected delay rate spikes detected.")